# CSC316 — NBA Late-Game Strategy (Five Real-Data Charts)

**Dataset:** SportsDataverse / hoopR, ESPN NBA play-by-play, **2024–25 regular season**.

This notebook recreates the five visualizations selected in your process book: **J1, J4-R, S4, J2, H4**.

**Important:** SportsDataverse's data is collected from ESPN's NBA feed, *not* directly from `stats.nba.com`. If your process book names NBA.com, update the attribution.

Run **Runtime → Run all** in Google Colab. The first run downloads an entire season (could be a large file).

In [ ]:
%pip -q install --upgrade numpy pandas matplotlib pyarrow

## 1. Data cleaning and plotting code

The cell below defines the five plot functions. No results are fabricated or hard-coded.

In [ ]:
"""CSC316 NBA play-by-play: five selected visualizations (J1, J4-R, S4, J2, H4).

Run:
  python -m pip install pandas numpy matplotlib pyarrow
  python nba_five_graphs.py --season 2025 --out figures
  python nba_five_graphs.py --input play_by_play_2025.parquet --out figures

Data source: SportsDataverse hoopR ESPN NBA play-by-play public season releases.
Season year is the ENDING year; 2025 is 2024-25.
No synthetic results are included in the output.
"""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle, Polygon

DATA_URL = ('https://github.com/sportsdataverse/sportsdataverse-data/releases/'
            'download/espn_nba_pbp/play_by_play_{season}.parquet')
DEF_LABELS = ['1–3', '4–6', '7–10', '11+']
TIME_LABELS = ['9–12', '6–9', '3–6', '0–3']
DEF_BINS = [0, 3, 6, 10, np.inf]
TIME_BINS = [0, 180, 360, 540, 720.001]
TIME_ASC_LABELS = ['0–3', '3–6', '6–9', '9–12']


def load_data(path: str|None, season: int) -> pd.DataFrame:
    if path:
        print(f'Loading local play-by-play file: {path}', flush=True)
        if path.lower().endswith('.csv'):
            return pd.read_csv(path, low_memory=False)
        return pd.read_parquet(path)
    url = DATA_URL.format(season=season)
    print(f'Downloading full season play-by-play from:\n{url}', flush=True)
    return pd.read_parquet(url)


def convert_bool(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    return s.astype(str).str.lower().isin({'true','1','t','yes'})


def prep(raw: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    keys = ['game_id','game_play_number','period_number','clock_display_value','home_score',
            'away_score','home_team_id','away_team_id','team_id','shooting_play',
            'points_attempted','scoring_play','season_type','home_team_abbrev','away_team_abbrev']
    missing = sorted(set(keys)-set(raw.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}. This script expects the SportsDataverse ESPN PBP schema.')
    df=raw.copy()
    df['season_type']=pd.to_numeric(df['season_type'],errors='coerce')
    df=df.loc[df['season_type'].eq(2)].copy()  # regular season only
    if df.empty: raise ValueError('No regular season rows (season_type=2) found.')
    for col in ['home_score','away_score','home_team_id','away_team_id','team_id',
                'points_attempted','period_number','game_play_number']:
        df[col]=pd.to_numeric(df[col],errors='coerce')
    for col in ['shooting_play','scoring_play']:
        df[col]=convert_bool(df[col])
    df = df.dropna(subset=['game_id','game_play_number','home_team_id','away_team_id']).copy()
    df=df.sort_values(['game_id','game_play_number'],kind='stable').reset_index(drop=True)
    # Derived PRE-event score (correct for shots, which change score on the same event).
    df['home_pre']=df.groupby('game_id',sort=False)['home_score'].shift(1).fillna(0)
    df['away_pre']=df.groupby('game_id',sort=False)['away_score'].shift(1).fillna(0)
    clock=df['clock_display_value'].astype(str).str.extract(r'(?P<minute>\d{1,2}):(?P<second>\d{1,2}(?:\.\d+)?)')
    df['seconds_left']=pd.to_numeric(clock['minute'],errors='coerce')*60+pd.to_numeric(clock['second'],errors='coerce')
    validteam = df['team_id'].eq(df['home_team_id']) | df['team_id'].eq(df['away_team_id'])
    df['valid_event_team']=validteam
    df['deficit_pre']=np.where(df['team_id'].eq(df['home_team_id']),
                               df['away_pre']-df['home_pre'],df['home_pre']-df['away_pre'])
    df.loc[~validteam,'deficit_pre']=np.nan
    df['team_abbrev']=np.where(df['team_id'].eq(df['home_team_id']),df['home_team_abbrev'],df['away_team_abbrev'])
    df['is_fga']=df['shooting_play'] & df['points_attempted'].isin([2,3]) & validteam
    df['is_three']=df['is_fga'] & df['points_attempted'].eq(3)
    df['is_made']=df['is_fga'] & df['scoring_play']
    typ=df['type_text'].astype(str).str.lower() if 'type_text' in df else pd.Series('',index=df.index)
    df['is_turnover']=typ.str.contains('turnover',na=False) & validteam
    # ESPN text is not perfectly uniform; verify if turnover codes are present.
    last = df.groupby('game_id',sort=False).tail(1).copy()
    last['final_winner_id']=np.where(last['home_score']>last['away_score'],last['home_team_id'],last['away_team_id'])
    last['is_final_valid']=(last['home_score'].ge(60) & last['away_score'].ge(60) & last['home_score'].ne(last['away_score']))
    last=last[last['is_final_valid']].drop_duplicates('game_id')
    if last.empty: raise ValueError('No games with plausible final scores found.')
    fin=last[['game_id','final_winner_id','home_score','away_score']].rename(columns={'home_score':'final_home','away_score':'final_away'})
    df=df[df['game_id'].isin(fin['game_id'])].copy()
    df=df.merge(fin[['game_id','final_winner_id']],on='game_id',how='left',validate='many_to_one')
    q4=df[df['period_number'].eq(4)&df['seconds_left'].between(0,720)].copy()
    q4['time_band']=pd.cut(q4['seconds_left'],TIME_BINS,labels=TIME_ASC_LABELS,include_lowest=True)
    q4['def_band']=pd.cut(q4['deficit_pre'],DEF_BINS,labels=DEF_LABELS,include_lowest=False)
    trailing=q4[(q4['deficit_pre']>0)&q4['valid_event_team']].copy()
    print(f'Regular-season games: {fin.game_id.nunique():,}; total events: {len(df):,}; '
          f'Q4 trailing field-goal attempts: {int(trailing.is_fga.sum()):,}',flush=True)
    return df,q4,trailing


def save(fig, out:Path, filename:str):
    fig.savefig(out/filename, dpi=190, bbox_inches='tight',facecolor='white')
    plt.close(fig)


def graph1(trailing:pd.DataFrame,out:Path)->dict:
    shots=trailing[trailing['is_fga']].copy()
    g=shots.groupby(['time_band','def_band'],observed=True).agg(n=('is_fga','size'),n3=('is_three','sum')).reset_index()
    g['pct']=100*g['n3']/g['n']
    fig,ax=plt.subplots(figsize=(10,6))
    for t in TIME_LABELS:
        sub=g[g.time_band.astype(str).eq(t)].set_index('def_band').reindex(DEF_LABELS)
        ax.plot(DEF_LABELS,sub['pct'].values,marker='o',linewidth=2,label=f'{t} min left')
    ax.set(xlabel='Trailing margin before shot (points)',ylabel='Three-point share of all field-goal attempts (%)',
           title='J1 · When do trailing teams turn to three-pointers?')
    ax.grid(alpha=.2)
    ax.legend(title='Q4 game clock',frameon=False)
    ax.set_ylim(bottom=0,top=100)
    fig.text(.10,.015,'2024–25 NBA regular season  ·  Q4 only  ·  Each shot uses score before the attempt',fontsize=9)
    save(fig,out,'01_J1_three_point_share.png')
    g.to_csv(out/'01_J1_data.csv',index=False)
    return {'n_fga':len(shots)}


def exposures_and_states(q4:pd.DataFrame,df:pd.DataFrame):
    # Between two events at consecutive clock times, the post-event score is in force.
    e=q4.copy()
    e['next_sec']=e.groupby('game_id',sort=False)['seconds_left'].shift(-1)
    e['next_sec']=e['next_sec'].fillna(0)
    # Clamp invalid / duplicate clocks to zero exposure.
    e['next_sec']=np.minimum(e['seconds_left'],e['next_sec'])
    e['deficit_after']=abs(e['home_score']-e['away_score'])
    e['trailing_team_id']=np.where(e['home_score']<e['away_score'],e['home_team_id'],
                                    np.where(e['away_score']<e['home_score'],e['away_team_id'],np.nan))
    e['def_band_after']=pd.cut(e['deficit_after'],DEF_BINS,labels=DEF_LABELS)
    exposure=[]
    for time_label,low,high in [('0–3',0,180),('3–6',180,360),('6–9',360,540),('9–12',540,720)]:
        time_delta=(np.minimum(e['seconds_left'],high)-np.maximum(e['next_sec'],low)).clip(lower=0)
        frame=e.loc[(time_delta>0)&e['def_band_after'].notna()&e['trailing_team_id'].notna(),['def_band_after']].copy()
        frame['team_seconds']=time_delta.loc[frame.index]
        frame['time_band']=time_label
        exposure.append(frame)
    exposure=pd.concat(exposure,ignore_index=True).groupby(['def_band_after','time_band'],observed=True)['team_seconds'].sum().rename('trailing_team_seconds').reset_index()
    # Each trailing team-game is counted at most once in a (time, deficit) state.
    states=e[e['trailing_team_id'].notna()&e['def_band_after'].notna()].copy()
    states['time_band']=pd.cut(states['seconds_left'],TIME_BINS,labels=TIME_ASC_LABELS,include_lowest=True)
    states=states.drop_duplicates(['game_id','trailing_team_id','def_band_after','time_band'])
    states['comeback']=states['trailing_team_id'].eq(states['final_winner_id'])
    state_stats=states.groupby(['def_band_after','time_band'],observed=True).agg(
        comeback_rate=('comeback','mean'),n_state_games=('comeback','size')).reset_index()
    return exposure,state_stats


def graph2(q4:pd.DataFrame,trailing:pd.DataFrame,out:Path)->dict:
    exposures,states=exposures_and_states(q4,None)
    measures=trailing.groupby(['def_band','time_band'],observed=True).agg(
        fga=('is_fga','sum'), threes=('is_three','sum'), turnovers=('is_turnover','sum')).reset_index().rename(columns={'def_band':'def_band_after'})
    g=exposures.merge(measures,on=['def_band_after','time_band'],how='left').merge(states,on=['def_band_after','time_band'],how='left')
    for col in ['fga','threes','turnovers']: g[col]=g[col].fillna(0)
    g['three_share']=np.where(g.fga>0,g.threes/g.fga,np.nan)
    g['fga_per_min']=g.fga/(g.trailing_team_seconds/60)
    g['turnover_per_min']=g.turnovers/(g.trailing_team_seconds/60)
    # The three spokes have different units. Rescale within the plotted cells.
    cols=['three_share','fga_per_min','turnover_per_min']
    maxvals=np.array([g[c].max(skipna=True) for c in cols],dtype=float)
    maxvals=np.where(maxvals>0,maxvals,1)
    fig,axs=plt.subplots(4,4,figsize=(12.3,12.1),subplot_kw=dict(aspect='equal'))
    angles=np.deg2rad([90,210,330])
    for i,d in enumerate(DEF_LABELS):
        for j,t in enumerate(TIME_LABELS):
            ax=axs[i,j];ax.set_xlim(-1.14,1.14);ax.set_ylim(-1.24,1.18);ax.axis('off')
            for rr in [.38,.73]: ax.add_patch(Circle((0,0),rr,fill=False,linewidth=.65,alpha=.25))
            for ang in angles: ax.plot([0,.79*np.cos(ang)],[0,.79*np.sin(ang)],linewidth=.75,alpha=.5)
            sub=g[(g.def_band_after.astype(str)==d)&(g.time_band.astype(str)==t)]
            if not len(sub):
                ax.text(0,-.05,'No data',ha='center',fontsize=10)
                continue
            r=sub.iloc[0]
            vals=np.array([r[c] for c in cols],dtype=float)/maxvals
            vals=np.where(np.isfinite(vals),np.clip(vals,0,1),0)
            coords=np.c_[.10+.64*vals]*np.c_[np.cos(angles),np.sin(angles)]
            ax.add_patch(Polygon(coords,closed=True,alpha=.34,edgecolor='black',linewidth=1.3))
            ax.plot(np.r_[coords[:,0],coords[0,0]],np.r_[coords[:,1],coords[0,1]],lw=1.2)
            comeback=float(r.comeback_rate) if pd.notna(r.comeback_rate) else 0
            ax.add_patch(Arc((0,0),1.9,1.9,theta1=90,theta2=90+360*np.clip(comeback,0,1),lw=3))
            ax.text(0,-1.12,f'3PA {r.three_share:.0%} · win {comeback:.0%}',ha='center',fontsize=8)
            if i==0: ax.set_title(t+' min left',fontsize=12,fontweight='bold',pad=9)
            if j==0: ax.text(-1.33,0,d+' pts',ha='right',va='center',fontsize=11,fontweight='bold')
    fig.suptitle('J4-R · Game-state tactical fingerprint',fontsize=18,y=.995)
    fig.text(.5,.945,'Spokes: 3PA share (up), field-goal attempts per minute (lower left), turnovers per minute (lower right)',ha='center',fontsize=10)
    fig.text(.5,.923,'Spoke lengths scaled to each metric’s maximum across the grid  ·  Outer arc: historical comeback rate',ha='center',fontsize=9)
    fig.text(.5,.028,'Q4 trailing-team exposure based on time between play events  ·  Win rate uses unique team-games per state cell',ha='center',fontsize=9)
    fig.subplots_adjust(left=.10,right=.97,top=.89,bottom=.065,hspace=.34,wspace=.10)
    save(fig,out,'02_J4R_tactical_fingerprint.png')
    g.to_csv(out/'02_J4R_data.csv',index=False)
    return {'cells_with_data':len(g),'note':'Time between events is assigned to the score state immediately after the first event.'}


def graph3(trailing:pd.DataFrame,out:Path)->dict:
    shots=trailing[trailing.is_fga].copy()
    g=shots.groupby(['def_band','time_band'],observed=True).agg(n=('is_fga','size'),made=('is_made','sum')).reset_index()
    g['fg_pct']=100*g['made']/g['n']
    m=g.pivot(index='def_band',columns='time_band',values='fg_pct').reindex(index=DEF_LABELS,columns=TIME_LABELS)
    n=g.pivot(index='def_band',columns='time_band',values='n').reindex(index=DEF_LABELS,columns=TIME_LABELS)
    fig,ax=plt.subplots(figsize=(10,6))
    img=ax.imshow(m.to_numpy(dtype=float),cmap='viridis',vmin=0,vmax=70,aspect='auto')
    for i in range(4):
        for j in range(4):
            val=m.iloc[i,j];count=n.iloc[i,j]
            if pd.notna(val):
                ax.text(j,i,f'{val:.1f}%\n(n={count:,.0f})',ha='center',va='center',fontsize=11,
                        color='white' if val<34 else 'black',fontweight='bold')
    ax.set(xticks=range(4),xticklabels=TIME_LABELS,yticks=range(4),yticklabels=DEF_LABELS,
           xlabel='Minutes remaining in Q4',ylabel='Score deficit before shot (points)',
           title='S4 · Shooting efficiency when trailing')
    cbar=fig.colorbar(img,ax=ax,shrink=.87);cbar.set_label('Field-goal percentage (%)')
    fig.text(.1,.015,'FG% = made 2-point and 3-point attempts / all 2-point and 3-point attempts · Free throws excluded',fontsize=9)
    save(fig,out,'03_S4_shooting_accuracy_heatmap.png')
    g.to_csv(out/'03_S4_data.csv',index=False)
    return {'n_fga':len(shots)}


def graph4(q4:pd.DataFrame,df:pd.DataFrame,out:Path)->dict:
    # Select the team trailing close to exactly five minutes in Q4.
    q=q4[q4.seconds_left.between(300,345)].copy()
    snaps=q.groupby('game_id',sort=False).tail(1).copy()  # latest event before reaching 5:00
    snaps['five_min_deficit']=(snaps.home_score-snaps.away_score).abs()
    snaps=snaps[snaps.five_min_deficit.between(1,20)].copy()
    snaps['team_at_risk']=np.where(snaps.home_score<snaps.away_score,snaps.home_team_id,snaps.away_team_id)
    snaps['comeback']=snaps.team_at_risk.eq(snaps.final_winner_id)
    shots=q4[q4.is_fga].copy()
    shots['window']=np.select([shots.seconds_left.between(0,300,inclusive='left'),
                               shots.seconds_left.between(300,600,inclusive='left')],['last5','prior5'],default='ignore')
    shots=shots[shots.window.ne('ignore')]
    teamshots=snaps[['game_id','team_at_risk']].merge(shots[['game_id','team_id','window']],
                                                     left_on=['game_id','team_at_risk'],right_on=['game_id','team_id'],how='left')
    ct=teamshots[teamshots.window.isin(['last5','prior5'])].groupby(['game_id','window']).size().unstack(fill_value=0)
    for col in ['last5','prior5']:
        if col not in ct: ct[col]=0
    snaps=snaps.merge(ct[['last5','prior5']],left_on='game_id',right_index=True,how='left').fillna({'last5':0,'prior5':0})
    snaps['pace_change']=(snaps.last5-snaps.prior5)/5  # attempted shots per team-minute
    snaps['def_group']=pd.cut(snaps.five_min_deficit,[0,5,10,20],labels=['1–5','6–10','11–20'])
    # Pre-specified (non-data-adaptive) 0.2 FGA/min bins so the comparison is reproducible.
    edges=np.r_[-np.inf,np.arange(-1.2,1.41,.2),np.inf]
    snaps['pace_bin']=pd.cut(snaps.pace_change,edges)
    g=snaps.groupby(['def_group','pace_bin'],observed=True).agg(
        n=('comeback','size'),comebacks=('comeback','sum'),x=('pace_change','mean')).reset_index()
    g['win_pct']=100*g.comebacks/g.n
    g=g[g.n>=8]   # suppress very unstable tiny cells
    fig,ax=plt.subplots(figsize=(10,6))
    for d in ['1–5','6–10','11–20']:
        s=g[g.def_group.astype(str).eq(d)]
        ax.scatter(s.x,s.win_pct,s=np.maximum(50,s.n*6),alpha=.65,label=d+' behind',edgecolor='black',linewidth=.5)
    ax.axvline(0,linestyle='--',linewidth=1,color='gray')
    ax.set(xlabel='Change in field-goal attempts per team-minute (last 5 min − previous 5 min)',
           ylabel='Historical comeback rate (%)',
           title='J2 · Does speeding up correspond to more comebacks?')
    ax.legend(title='Deficit with 5 min left',frameon=False)
    ax.grid(alpha=.2)
    fig.text(.1,.015,'One observation per game trailing near 5:00 · Bubble size ∝ sample games · Associations are not causal',fontsize=9)
    save(fig,out,'04_J2_pace_change_comeback.png')
    g.to_csv(out/'04_J2_aggregated_data.csv',index=False)
    snaps[['game_id','five_min_deficit','pace_change','last5','prior5','comeback']].to_csv(out/'04_J2_game_level_data.csv',index=False)
    return {'n_teams_trailing_5min':len(snaps),'cells_plotted':len(g)}


def graph5(trailing:pd.DataFrame,out:Path)->dict:
    shots=trailing[trailing.is_fga & trailing.seconds_left.le(300) & trailing.deficit_pre.between(6,10)].copy()
    g=shots.groupby('team_abbrev').agg(n=('is_fga','size'),threes=('is_three','sum')).reset_index()
    g['pct']=100*g.threes/g.n
    g=g[g.n>=25].sort_values('pct',ascending=True)
    if g.empty:
        raise ValueError('H4 team chart has no teams meeting minimum sample (25) in this dataset.')
    league=100*shots.is_three.sum()/len(shots)
    fig,ax=plt.subplots(figsize=(10,8))
    ax.barh(g.team_abbrev,g.pct,alpha=.8)
    ax.axvline(league,linestyle='--',linewidth=1.3,label=f'League average: {league:.1f}%')
    for y,(_,r) in enumerate(g.iterrows()):
        ax.text(min(r.pct+1,97),y,f'{r.pct:.1f}% (n={int(r.n)})',va='center',fontsize=8)
    ax.set(xlim=(0,100),xlabel='Share of field-goal attempts that are three-pointers (%)',
           ylabel='NBA team',title='H4 · Which teams take more threes in similar late-game situations?')
    ax.legend(frameon=False,loc='lower right')
    ax.grid(axis='x',alpha=.2)
    fig.text(.1,.01,'Same game state for all teams: trailing by 6–10 points, with ≤5:00 left in Q4 · At least 25 attempts/team',fontsize=9)
    save(fig,out,'05_H4_team_comparison.png')
    g.to_csv(out/'05_H4_data.csv',index=False)
    return {'n_teams_displayed':len(g),'league_three_share':round(league,4)}


def main(argv=None):
    p=argparse.ArgumentParser(description='Generate five CSC316 NBA charts from real play-by-play data.')
    p.add_argument('--season',type=int,default=2025,help='Ending season year (2025 = 2024–25)')
    p.add_argument('--input',help='Local .csv or .parquet instead of downloading')
    p.add_argument('--out',default='nba_charts')
    args=p.parse_args(argv)
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    raw=load_data(args.input,args.season)
    df,q4,trailing=prep(raw)
    results={'source':args.input or DATA_URL.format(season=args.season),'season_end_year':args.season,
             'regular_season_games':df.game_id.nunique(),'q4_events':len(q4),
             'trailing_q4_fga':int(trailing.is_fga.sum())}
    for title,func,arguments in [
        ('J1',graph1,(trailing,out)),('J4-R',graph2,(q4,trailing,out)),
        ('S4',graph3,(trailing,out)),('J2',graph4,(q4,df,out)),
        ('H4',graph5,(trailing,out))]:
        results[title]=func(*arguments)
        print(f'Saved {title}',flush=True)
    (out/'analysis_metadata.json').write_text(json.dumps(results,indent=2,default=str),encoding='utf8')
    print(f'Finished. Five PNG figures and supporting CSV summaries in {out.resolve()}',flush=True)
    return results




## 2. Download season data and produce graphs

**2025** means the season ending in 2025 (2024–25). Use `--season 2024` for 2023–24.

In [ ]:
results = main(['--season', '2025', '--out', 'nba_charts'])
results

## 3. Show all five graphs

In [ ]:
from IPython.display import display, Image
from pathlib import Path

for fn in sorted(Path('nba_charts').glob('*.png')):
    print(fn.name)
    display(Image(filename=str(fn), width=850))

## 4. Export PNGs, source tables, and analysis metadata

In [ ]:
import shutil
shutil.make_archive('CSC316_NBA_5_RealData_Graphs', 'zip', 'nba_charts')
print('Saved CSC316_NBA_5_RealData_Graphs.zip')
# If using Google Colab:
try:
    from google.colab import files
    files.download('CSC316_NBA_5_RealData_Graphs.zip')
except ImportError:
    print('Download the ZIP from the notebook working directory.')

## Interpretation reminders
- Charts are descriptive, not evidence of a causal effect of strategy on outcomes.
- Read the `*.csv` tables and `analysis_metadata.json` before drafting your 2–3 insights.
- Check games, sample sizes and definitions before using statistics in your graded process book.
- If the automated download fails, manually download `play_by_play_2025.parquet` from the SportsDataverse link and run `main(['--input', '/content/play_by_play_2025.parquet', '--out', 'nba_charts'])`.